In [6]:
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("twitter_transformation").getOrCreate()

df = spark.read.json("/home/guiandreis/Airflow-nodocker-Spark/data_lake/twitter_posts_raw/")


import pyspark.sql.functions as f 



from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "created_at",
    to_timestamp("created_at", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSZ")
)

from pyspark.sql.functions import col , size

df = df.select(
    "*",
    col("public_metrics.like_count").alias("like_count"),
    col("public_metrics.quote_count").alias("quote_count"),
    col("public_metrics.reply_count").alias("reply_count"),
    col("public_metrics.retweet_count").alias("retweet_count")
).drop("public_metrics")

df = df.withColumn("edit_count", size(col("edit_history_tweet_ids")))


df = df.withColumn(
    "engagement",
    col("like_count") +
    col("retweet_count") +
    col("reply_count") +
    col("quote_count")
)


from pyspark.sql.functions import to_date

df = df.withColumn("date", to_date("created_at"))

df.show(6)

 #% de replies vs tweets 

+---------+---------------+--------------------+----------------------+---+-------------------+----+--------------------+----------+-----------+-----------+-------------+----------+----------+----------+
|author_id|conversation_id|          created_at|edit_history_tweet_ids| id|in_reply_to_user_id|lang|                text|like_count|quote_count|reply_count|retweet_count|edit_count|engagement|      date|
+---------+---------------+--------------------+----------------------+---+-------------------+----+--------------------+----------+-----------+-----------+-------------+----------+----------+----------+
|       26|             95|2026-02-09 15:21:...|                   [3]| 29|                 26|  en|Tweet fictício ge...|        84|         13|         74|           32|         1|       203|2026-02-09|
|        8|             85|2026-02-08 23:56:...|                  [50]| 73|                 24|  en|Outro tweet fictí...|        19|         96|         19|           16|         1|   

In [12]:

from pyspark.sql.functions import col
from pyspark.sql.functions import to_date
from pyspark.sql.functions import current_date

df = spark.read.json("/home/guiandreis/Airflow-nodocker-Spark/data_lake/twitter_posts_raw/")


def get_tweets_data(df):
        
    tweet_df_raw = df.select(
        "*",
        col("public_metrics.like_count").alias("like_count"),
        col("public_metrics.quote_count").alias("quote_count"),
        col("public_metrics.reply_count").alias("reply_count"),
        col("public_metrics.retweet_count").alias("retweet_count")
    ).drop("public_metrics")
    
    return tweet_df_raw

def adjusting_columns(tweet_df_raw):
    new_cols = [
    "user_id",
    "thread_id",
    "created_at",
    "edit_versions_ids",
    "tweet_id",
    "reply_to_user_id",
    "language",
    "tweet_text",
    "likes",
    "quotes",
    "replies",
    "retweets"
]

    tweet_df= tweet_df_raw.toDF(*new_cols)
    tweet_df = tweet_df.withColumn("created_date", to_date("created_at"))
    tweet_df_clean = tweet_df.withColumn("processing_date", current_date())
    
    return tweet_df_clean
    
def load_parquet(tweet_df_clean, path_to_save="/home/guiandreis/Airflow-nodocker-Spark/data/silver"):
    tweet_df_clean.write.mode("overwrite") \
    .partitionBy("created_date") \
    .parquet(path_to_save)
    
def run(spark,  path_to_save, src="/home/guiandreis/Airflow-nodocker-Spark/data_lake/twitter_posts_raw/"):
    df = spark.read.json(src)
    tweet_df_raw = get_tweets_data(df)
    tweet_df_clean = adjusting_columns(tweet_df_raw)
    tweet_df_clean.show(6, truncate=False)
    return load_parquet(tweet_df_clean) 

run(spark = SparkSession.builder.appName("twitter_transformation").getOrCreate() ,path_to_save="/home/guiandreis/Airflow-nodocker-Spark/data/silver")


+-------+---------+-------------------------------+-----------------+--------+----------------+--------+--------------------------------------------------------+-----+------+-------+--------+------------+---------------+
|user_id|thread_id|created_at                     |edit_versions_ids|tweet_id|reply_to_user_id|language|tweet_text                                              |likes|quotes|replies|retweets|created_date|processing_date|
+-------+---------+-------------------------------+-----------------+--------+----------------+--------+--------------------------------------------------------+-----+------+-------+--------+------------+---------------+
|26     |95       |2026-02-09T18:21:48.152645+0000|[3]              |29      |26              |en      |Tweet fictício gerado automaticamente sobre data science|84   |13    |74     |32      |2026-02-09  |2026-02-12     |
|8      |85       |2026-02-09T02:56:42.904831+0000|[50]             |73      |24              |en      |Outro tweet 

26/02/12 08:33:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/12 08:33:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/12 08:33:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/12 08:33:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/12 08:33:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/12 08:33:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/12 08:33:21 WARN MemoryManager: Total allocation exceeds 95.0